# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library. The dataset follows the FAIR principles and offers access to protein abundance and modification information in human mast cell extracellular vesicle samples.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Cite As: {getattr(metadata, 'cite_as', getattr(metadata, 'citeAs', None))}\n")
print(f"Keywords: {', '.join(metadata.keywords) if hasattr(metadata, 'keywords') else ''}\n")
print(f"License: {metadata.license}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs. All Croissant entities are referenced by their `@id` fields.

Let's enumerate the available record sets and for each, print its `@id` and included field `@id`s.

In [ ]:
# Display all record sets and their field @id
record_sets = list(dataset.record_sets)
print(f"Available Record Sets ({len(record_sets)}):\n")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    elif not isinstance(fields, list):
        fields = []
    for f in fields:
        fid = f['@id'] if isinstance(f, dict) and '@id' in f else str(f)
        print(f"    - Field @id: {fid}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For this example, we select the primary protein abundance data record set, which often contains quantitative measurement columns. Replace `<record_set_id>` below with the desired record set's `@id`.

In [ ]:
# Manually select the main record set @id after viewing the previous cell output.
# E.g., suppose the table-like record set has @id 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'
# Replace as appropriate for your schema.
primary_record_set = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'
all_record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for record_set_id in all_record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            dataframes[record_set_id] = pd.DataFrame(records)
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

print(f"Loaded record sets with non-empty dataframes: {list(dataframes.keys())}\n")

if primary_record_set in dataframes:
    print(f"Columns in primary record set ({primary_record_set}):")
    print(dataframes[primary_record_set].columns.tolist())
    display(dataframes[primary_record_set].head())
else:
    print(f"Primary record set '{primary_record_set}' was not loaded as a DataFrame.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All fields are referenced by their column/field `@id` as per the Croissant specification.

We will demonstrate filtering by a numeric field, normalizing it, and grouping by a categorical field.

In [ ]:
import numpy as np

# Example: Select a numeric field by its @id (e.g. 'Abundance' or 'Coverage') as shown in previous column list
# For demonstration, let's assume field '@id': 'Abundance_Sample_A' (replace as actual from previous cell)
numeric_field_id = None
df = dataframes.get(primary_record_set)

# Guess at a likely numeric field to analyze based on available columns
if df is not None:
    for col in df.columns:
        # Pick a column commonly representing abundance
        if 'abundance' in col.lower() or 'coverage' in col.lower() or 'count' in col.lower():
            numeric_field_id = col
            break

if numeric_field_id is not None and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].quantile(0.75)
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} above 75th percentile ({threshold}):")
    display(filtered_df.head())

    # Normalizing
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a likely categorical field (e.g., 'Description' or 'accession' or another column)
    group_field_id = None
    for col in df.columns:
        if col.lower() in ['description', 'accession', 'protein']:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field detected for analysis. Please review the previous column list and edit the notebook to specify an appropriate numeric field @id.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll plot a histogram of the selected numeric field and, if grouping is possible, display the top groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouped_df exists, show bar plot of top 10
    if 'grouped_df' in locals() or 'grouped_df' in globals():
        plt.figure(figsize=(10, 5))
        plot_df = grouped_df.head(10)
        sns.barplot(data=plot_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Top 10 {group_field_id} by mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field or dataframe to visualize.")

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and explore the dataset "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells." Following the Croissant schema, we referenced all entities by their `@id` fields, loaded the available record sets, extracted tabular data, performed basic EDA such as filtering and normalization, and visualized the results. This workflow can be extended to more advanced analyses, integrating field documentation and metadata to guarantee reproducible research.